# 03 - Kiểm thử Module Segmentation
Kiểm tra việc huấn luyện tìm ngưỡng tối ưu (Binary Search & Gaussian) và thuật toán phân đoạn kèm lọc khoảng lặng ảo.


In [4]:
import sys
from pathlib import Path
import numpy as np

ROOT_DIR = Path.cwd().resolve()
while ROOT_DIR.name != 'main_gk' and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.audio.dataset import load_dataset
from src.segmentation.threshold import calibrate_thresholds
from src.segmentation.segment import segment_signal


### 1. Huấn luyện tìm ngưỡng trên tập TinHieuHuanLuyen


In [5]:
train_signals = load_dataset(ROOT_DIR / 'TinHieuHuanLuyen')
thresholds = calibrate_thresholds(train_signals)

print('=== KẾT QUẢ HUẤN LUYỆN NGƯỠNG T ===')
print(f'BINARY SEARCH (Hodgkinson 2012):')
print(f'  - Global  : {thresholds.binary_threshold:.6f}')
print(f'  - Phone   : {thresholds.binary_threshold_phone:.6f}')
print(f'  - Studio  : {thresholds.binary_threshold_studio:.6f}')
print(f'GAUSSIAN MODEL:')
print(f'  - Phone   : {thresholds.gaussian_threshold_phone:.6f}')
print(f'  - Studio  : {thresholds.gaussian_threshold_studio:.6f}')
print(f'  - Mean Sp / Sil: {thresholds.mean_speech:.4f} / {thresholds.mean_silence:.4f}')


=== KẾT QUẢ HUẤN LUYỆN NGƯỠNG T ===
BINARY SEARCH (Hodgkinson 2012):
  - Global  : 0.000798
  - Phone   : 0.000826
  - Studio  : 0.000644
GAUSSIAN MODEL:
  - Phone   : 0.001148
  - Studio  : 0.000414
  - Mean Sp / Sil: 0.2025 / 0.0002


### 2. Kiểm tra phân đoạn và lọc khoảng lặng ảo 200ms trên 1 file test


In [6]:
test_signals = load_dataset(ROOT_DIR / 'TinHieuKiemThu')
sig = test_signals[2]  # studio_F2

th_val = thresholds.get_threshold('binary', sig.is_phone)
seg_res = segment_signal(sig, threshold=th_val, algorithm='binary', min_silence_ms=200.0)

print(f'File: {sig.name} (is_phone={sig.is_phone})')
print(f'Ngưỡng sử dụng           : {th_val:.6f}')
print(f'Biên chuẩn Ground Truth  : {sig.gt_boundaries}')
print(f'Biên dự đoán thuật toán  : {seg_res.predicted_boundaries}')
print(f'Danh sách đoạn phân loại :')
for s in seg_res.segments:
    print(f'  [{s[0]:.2f}s -> {s[1]:.2f}s]: {s[2]}')


File: studio_F2 (is_phone=False)
Ngưỡng sử dụng           : 0.000644
Biên chuẩn Ground Truth  : [0.77, 2.37]
Biên dự đoán thuật toán  : [0.77, 2.37]
Danh sách đoạn phân loại :
  [0.01s -> 0.77s]: silence
  [0.77s -> 2.37s]: speech
  [2.37s -> 3.13s]: silence
